In [ ]:
import sys
import threading
import time

: 

In [2]:
def check_gil_status():
    """Checks if the GIL is currently active in this runtime."""
    # Check if the internal check function exists (Python 3.13+)
    if hasattr(sys, "_is_gil_enabled"):
        gil_enabled = sys._is_gil_enabled()
        status = "ENABLED (Threads run sequentially)" if gil_enabled else "DISABLED (TRUE MULTITHREADING!)"
    else:
        status = "ENABLED (Pre-Python 3.13 environment)"
    
    print(f"=== CURRENT RUNTIME STATUS ===")
    print(f"Python Version: {sys.version.split()[0]}")
    print(f"GIL Status:     {status}")
    print(f"=============================\n")

check_gil_status()

=== CURRENT RUNTIME STATUS ===
Python Version: 3.14.6
GIL Status:     DISABLED (TRUE MULTITHREADING!)



In [3]:
# ==========================================
# STEP 2: Define Shared State and Workers
# ==========================================
# Global variables for tracking
shared_counter = 0
counter_lock = threading.Lock()

def heavy_cpu_task(iterations, use_lock=True):
    """A math-heavy task that stresses a CPU core and increments a shared counter."""
    global shared_counter
    local_count = 0
    
    # Heavy CPU math work loop
    for i in range(iterations):
        _ = (i * 3) + (i ** 2)
        local_count += 1
        
    # Thread-safe counter update
    if use_lock:
        with counter_lock:
            shared_counter += local_count
    else:
        # DANGER: In free-threaded mode without a lock, this causes race conditions!
        shared_counter += local_count

def run_experiment(num_threads=4, iterations=5_000_000, use_lock=True):
    """Spawns parallel threads and benchmarks execution time."""
    global shared_counter
    shared_counter = 0  # Reset
    threads = []
    
    print(f"🚀 Spawning {num_threads} threads ({iterations:,} loops each)...")
    start_time = time.perf_counter()
    
    # Create and start threads
    for _ in range(num_threads):
        t = threading.Thread(target=heavy_cpu_task, args=(iterations, use_lock))
        threads.append(t)
        t.start()
        
    # Wait for completion
    for t in threads:
        t.join()
        
    end_time = time.perf_counter()
    total_time = end_time - start_time
    
    # Calculate expected vs actual values to detect data corruption
    expected = num_threads * iterations
    mismatch = "❌ CORRUPTED!" if shared_counter != expected else "✅ PERFECT Match"
    
    print(f"⏱️ Finished in: {total_time:.4f} seconds")
    print(f"📊 Counter State: Expected {expected:,} | Got {shared_counter:,} -> {mismatch}\n")


In [4]:
# ==========================================
# STEP 3: Run the Benchmarks
# ==========================================
# Experiment A: Thread-safe run (Standard control test)
print("--- RUN 1: Thread-Safe Processing ---")
run_experiment(num_threads=4, iterations=3_000_000, use_lock=True)

# Experiment B: Intentional Race Condition Test
# (Only breaks if GIL is actually Disabled!)
print("--- RUN 2: Dangerous Run (No Locks) ---")
run_experiment(num_threads=4, iterations=3_000_000, use_lock=False)

--- RUN 1: Thread-Safe Processing ---
🚀 Spawning 4 threads (3,000,000 loops each)...
⏱️ Finished in: 0.2494 seconds
📊 Counter State: Expected 12,000,000 | Got 12,000,000 -> ✅ PERFECT Match

--- RUN 2: Dangerous Run (No Locks) ---
🚀 Spawning 4 threads (3,000,000 loops each)...
⏱️ Finished in: 0.2016 seconds
📊 Counter State: Expected 12,000,000 | Got 12,000,000 -> ✅ PERFECT Match

